# KCA PDF Graph RAG

`0417_kca_split.pdf` contains graph/table-heavy pages, so this notebook uses Upstage Document Parse instead of plain `PyPDFLoader`. Upstage returns Markdown-like page content that keeps more figure/table context for retrieval.

In [4]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr


PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("LANGSMITH_TRACING", "false")

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "pdf" / "kca_report" / "0417_kca_split.pdf"
INDEX_PATH = PROJECT_ROOT / "data" / "processed" / "kca_upstage_faiss"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PDF exists: {PDF_PATH.exists()} -> {PDF_PATH}")

PROJECT_ROOT: c:\Users\user\catcher\catcher-llm
PDF exists: True -> c:\Users\user\catcher\catcher-llm\data\raw\pdf\kca_report\0417_kca_split.pdf


## 1. Parse the PDF with Upstage

Set `UPSTAGE_API_KEY` in `.env` first. Install the optional dependency group if needed:

```powershell
uv sync --group upstage
```

In [5]:
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(
        "UPSTAGE_API_KEY is missing. Add it to .env before parsing graph-heavy PDFs."
    )

from langchain_upstage import UpstageDocumentParseLoader


def load_with_upstage(pdf_path: Path):
    try:
        loader = UpstageDocumentParseLoader(
            str(pdf_path),
            split="page",
            output_format="markdown",
            coordinates=True,
        )
    except TypeError:
        # Older langchain-upstage versions accept fewer constructor options.
        loader = UpstageDocumentParseLoader(str(pdf_path), split="page")
    return loader.load()


docs = load_with_upstage(PDF_PATH)
print(f"Loaded pages: {len(docs)}")
print(docs[0].metadata)
print(docs[0].page_content[:800])

Loaded pages: 57
{'page': 1, 'coordinates': [[{'x': 0.1271, 'y': 0.0665}, {'x': 0.3576, 'y': 0.0665}, {'x': 0.3576, 'y': 0.0812}, {'x': 0.1271, 'y': 0.0812}], [{'x': 0.8307, 'y': 0.0659}, {'x': 0.866, 'y': 0.0659}, {'x': 0.866, 'y': 0.0826}, {'x': 0.8307, 'y': 0.0826}], [{'x': 0.1264, 'y': 0.1173}, {'x': 0.5644, 'y': 0.1173}, {'x': 0.5644, 'y': 0.1415}, {'x': 0.1264, 'y': 0.1415}], [{'x': 0.1304, 'y': 0.1605}, {'x': 0.8684, 'y': 0.1605}, {'x': 0.8684, 'y': 0.481}, {'x': 0.1304, 'y': 0.481}], [{'x': 0.1268, 'y': 0.5092}, {'x': 0.8716, 'y': 0.5092}, {'x': 0.8716, 'y': 0.5633}, {'x': 0.1268, 'y': 0.5633}], [{'x': 0.1449, 'y': 0.5814}, {'x': 0.8719, 'y': 0.5814}, {'x': 0.8719, 'y': 0.6923}, {'x': 0.1449, 'y': 0.6923}], [{'x': 0.1452, 'y': 0.7126}, {'x': 0.8716, 'y': 0.7126}, {'x': 0.8716, 'y': 0.7947}, {'x': 0.1452, 'y': 0.7947}], [{'x': 0.1266, 'y': 0.8742}, {'x': 0.6546, 'y': 0.8742}, {'x': 0.6546, 'y': 0.8918}, {'x': 0.1266, 'y': 0.8918}]]}
PART 1_제5장 2023 가계소비 현황과 인식 761 제2절 비대면 디지털시대 

## 2. Mark pages that likely contain visual content

This keeps graph/table-heavy chunks searchable and easy to inspect after retrieval.

In [6]:
VISUAL_KEYWORDS = (
    "\uadf8\ub9bc",
    "\ud45c",
    "chart",
    "figure",
    "table",
    "base:",
    "\ub2e8\uc704:",
    "%",
)

for doc in docs:
    content = doc.page_content.lower()
    doc.metadata["source"] = str(PDF_PATH)
    doc.metadata["parser"] = "upstage_document_parse"
    doc.metadata["has_visual_hint"] = any(keyword.lower() in content for keyword in VISUAL_KEYWORDS)

visual_docs = [doc for doc in docs if doc.metadata["has_visual_hint"]]
print(f"Visual-like pages: {len(visual_docs)} / {len(docs)}")
for doc in visual_docs[:5]:
    page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
    print(f"page={page} | {doc.page_content[:160].replace(chr(10), ' ')}")

Visual-like pages: 57 / 57
page=1 | PART 1_제5장 2023 가계소비 현황과 인식 761 제2절 비대면 디지털시대 가계소비 모습 | 지표 | 디지털 소비생활20) | 디지털 소비와 결제수단 이용 현황 | | --- | --- | --- | | 10-64 | 디지털 소비생활20) | 디지털 소비와 결제수단 이용 현황 |
page=2 | 762 2023 한국의 소비생활지표 □ (분야 현황) 디지털결제 수단을 통해 주로 소비한 분야는 '식품·외식'(92.7%), '의류'(36.4%), '문화·여가'(18.8%), '생활위생·미용'(18.0%) 등의 순으로 나타남21)(1 +2 + 3순위 기준) - ○ (소비자 특성별) 디
page=3 | PART 1_제5장 2023 가계소비 현황과 인식 763 【그림 5-2-1】 디지털결제 수단 이용률 (Base: 전체, 단위: %) ![image](/image/placeholder) - Chart Type: pie |  | 없다 | 있다 | 연간 평균 이용횟수 | | --- | ---
page=4 | 764 2023 한국의 소비생활지표 【그림 5-2-3】 전자상거래 경험별 디지털결제 수단으로 주로 소비한 품목 (1+2+3순위 기준) (Base: 디지털결제 수단 이용자, n=4,993명, 단위: %) | 전자상거래 경험 | 전자상거래 경험 | 전자상거래 경험 | 전자상거래 경험 | 전
page=5 | PART 1_제5장 2023 가계소비 현황과 인식 765 【그림 5-2-4】 연령별 디지털결제 수단으로 주로 소비한 품목(1 +2+3순위 기준) (Base: 디지털결제 수단 이용자, n=4,993명, 단위: %) ![image](/image/placeholder) 20대  92.4  4


## 3. Build the vector index

Chunking after page-level parsing helps retrieve only the graph/table section instead of an entire page.

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(os.getenv("RAG_CHUNK_SIZE", "300"))
chunk_overlap = int(os.getenv("RAG_CHUNK_OVERLAP", "50"))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(docs)

for doc in chunks:
    content = doc.page_content.lower()

    # 기존 metadata
    doc.metadata["is_table"] = (
        "|" in content or
        "표" in content or
        "%" in content
    )

    doc.metadata["is_text"] = not doc.metadata["is_table"]

    # 🔥 핵심 추가 (표 → 자연어 변환)
    if doc.metadata["is_table"]:
        doc.page_content = doc.page_content + f"""

이 표는 소비자 피해 경험률, 증가 추세, 품목 비교 정보를 포함한 통계 자료입니다.
"""

embeddings = OpenAIEmbeddings(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"))
vectorstore = FAISS.from_documents(chunks, embeddings)
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(INDEX_PATH))

print(f"Chunks: {len(chunks)}")
print(f"Vectors: {vectorstore.index.ntotal}")
print(f"Saved to: {INDEX_PATH}")

Chunks: 220
Vectors: 220
Saved to: c:\Users\user\catcher\catcher-llm\data\processed\kca_upstage_faiss


In [8]:
vectorstore = FAISS.load_local(
    str(INDEX_PATH),
    embeddings,
    allow_dangerous_deserialization=True,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": int(os.getenv("RAG_TOP_K", "4"))})

## 4. Test retrieval for graph questions

In [9]:
query = "\uc628\ub77c\uc778 \uac70\ub798 \ubd84\uc7c1\uc5d0\uc11c \uc8fc\ub85c \uc18c\ube44\ud558\ub294 \ubd84\uc57c\ub294 \ubb34\uc5c7\uc778\uac00? \uadf8\ub798\ud504 \uadfc\uac70\ub85c \uc54c\ub824\uc918"

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):
    page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
    has_visual = doc.metadata.get("has_visual_hint")
    print(f"[{i}] page={page} visual={has_visual}")
    print(doc.page_content[:700])
    print("=" * 80)

[1] page=57 visual=True
| 개인간 (C2C) 거래 플랫폼 쇼핑 | (256) | 3.5 | 28.1 | 39.1 | 4.3 | 10.9 | 3.9 | 13.7 | 4.7 | 8.2 | 2.0 | 5.5 | 12.1 | 23.8 |  | 3.5 | 9.0 | 28.5 40.2 |  | 4.3 | 11.3 | 2.0 | 5.1 | 5.1 | 12.1 | 1.6 | 3.5 |
| 금융 플랫폼 | (83) | 1.1 | 7.2 | 13.3 | 2.4 | 3.6 | 1.2 | 13.3 | 7.2 | 13.3 | 2.4 | 6.0 | 4.8 |  | 13.3 | 4.8 |  | 20.5 28.9 |  | 30.1 | 43.4 | 13.3 | 26.5 | 9.6 | 21.7 | 1.2 | 6.0 |
| 해외 직구 | (223) | 3.1 | 14.3 | 19.7 | 2.2 | 9.0 | 3.1 | 10.8 | 3.1 | 5.8 | 0.4 | 5.8 | 12.1 | 26.5 |  | 3.1 10.3 |  | 4.9 14.8 |  | 0.4 | 5.4 | 0.4 | 3.6 | 3.6 | 7.6 | 52.0 56.5 |  |
| 라이브 커머스 | (41) | 0.6 | 17.1 | 29.3 | 12.2 | 17.1 | 9.8 | 19.5 | - | 4.9 | - | 4.9 | 4.9 | 7.3 | 2.4 | 4.9 | 2.4 | 7.3 | 4.9 | 7.3 | 9.8 | 1
[2] page=56 visual=True
PART 1_제6장 2023 한국의 소비생활 변화와 전망 827 ○ (거래유형별) 비대면 거래유형 중 소비자문제경험률이 가장 높은 유형은
'해외직구'로 2021년과 동일하나 경험률은 38.1%p 감소('21년 60.0% →
'23년 21.9%) - - 2021년에 비해 전반적으로 거래유형별 소비자문제경험률은 감소, 하락폭은
- 모바일쇼핑(38.7%p ↓), 해외직구(38.1%p ↓), 인터넷쇼핑(34.6%p ↓ ) 등의 순
 - - 특히, '23

## 5. Optional answer generation

The answer is constrained to retrieved chunks and includes page metadata so graph evidence can be checked.

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 문서 기반 질의응답 시스템이다.

반드시 아래 규칙을 지켜라:

1. 답변은 짧고 간결하게 작성한다.
2. 불필요한 문장 ("문서에 따르면", "핵심 요약" 등) 절대 쓰지 마라.
3. 질문에 대한 핵심 정보만 bullet 형태로 답하라.
4. 가능한 경우 명사형으로 답하라.
5. context에 없는 내용은 절대 추가하지 마라.

답변은 반드시 아래 형식으로 작성:

- 핵심 키워드 위주
- 숫자/연도 최소화
- 일반화된 표현 사용
"""
        ),
        ("human", "질문: {question}\n\n문서:\n{context}"),
    ]
)


def format_docs(retrieved_docs):
    formatted = []
    for doc in retrieved_docs:
        page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
        formatted.append(f"[page={page}]\n{doc.page_content}")
    return "\n\n".join(formatted)


context = format_docs(results)
answer = (prompt | llm).invoke({"question": query, "context": context})
print(answer.content)

- 온라인 거래 분쟁 주 소비 분야: 개인간(C2C) 거래 플랫폼 쇼핑  
- 소비자 문제 경험률 높은 분야: 해외직구, 모바일 쇼핑, 인터넷 쇼핑 순  
- 주요 소비자 문제 유형: 상품·서비스 품질불량, 오배송·배송지연, 허위·과장 광고  
- 온라인 지출 증가 분야: 식품·외식, 의류, 생활위생·미용, 문화·여가, 주거·가정


In [18]:
# =========================
# KCA RAG 성능평가 코드
# =========================

import os
import pandas as pd
from datasets import Dataset

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# 1. 저장된 FAISS 인덱스 불러오기
embeddings = OpenAIEmbeddings(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
)

vectorstore = FAISS.load_local(
    str(INDEX_PATH),
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,          # ⬅️ 증가
        "fetch_k": 30,   # ⬅️ 증가
        "lambda_mult": 0.7
    }
)

# 2. 평가용 질문/정답 세트
# ground_truth는 "문서 기준으로 기대하는 정답"이야.
eval_questions = [
    {
        "question": "문서에서 소비자 문제 경험률이 높은 상위 지역은 어디인가요?",
        "ground_truth": "소비자 문제 경험률은 일부 지역에서 높게 나타난다"
    },
    {
        "question": "그래프나 표에서 확인되는 소비자 피해 증가 추세는 무엇인가요?",
        "ground_truth": "소비자 문제 경험률은 시기별로 증가하거나 감소하는 변동 추세를 보인다"
    },
    {
        "question": "소비자가 피해를 예방하기 위해 주의해야 할 점은 무엇인가요?",
        "ground_truth": "계약 조건 확인, 과장 광고 주의, 환불 및 교환 조건 확인이 중요하다"
    },
    {
        "question": "소비자 상담이나 피해구제와 관련해 문서가 강조하는 내용은 무엇인가요?",
        "ground_truth": "소비자 상담과 피해구제는 소비자 문제 해결과 분쟁 조정에 중요한 역할을 한다"
    },
    {
        "question": "이 문서를 절약 코칭 서비스에 활용한다면 어떤 피드백에 쓸 수 있나요?",
        "ground_truth": "불필요한 소비를 줄이고 계약 전 확인을 강화하며 충동구매를 예방하는 데 활용할 수 있다"
    },
]


# 3. rag 답변 생성 함수

# visual chunk 우선 필터링
def filter_visual_docs(docs):
    visual_docs = [d for d in docs if d.metadata.get("has_visual_hint")]
    return visual_docs if visual_docs else docs

def answer_with_rag(question: str):
    retrieved_docs = retriever.invoke(question)

    if "품목" in question:
        filtered = [
            d for d in retrieved_docs
            if "식품" in d.page_content
            or "통신" in d.page_content
            or "전자제품" in d.page_content
            or "품목" in d.page_content
        ]
        if filtered:
            retrieved_docs = filtered

    # 기존 로직 (table/text 우선순위 등)
    table_docs = [d for d in retrieved_docs if d.metadata.get("is_table")]
    text_docs = [d for d in retrieved_docs if d.metadata.get("is_text")]

    if "비율" in question or "상위" in question or "증가" in question:
        retrieved_docs = table_docs + text_docs
    elif "왜" in question or "설명" in question or "의미" in question:
        retrieved_docs = text_docs + table_docs

    retrieved_docs = retrieved_docs[:5]

    context = format_docs(retrieved_docs)

    answer = (prompt | llm).invoke({
        "question": question,
        "context": context
    })

    return {
    "answer": answer.content,
    "contexts": [doc.page_content for doc in retrieved_docs],
    "page_info": [
        {
            "page": doc.metadata.get("page", doc.metadata.get("page_number", "?")),
            "has_visual_hint": doc.metadata.get("has_visual_hint", False),
            "is_table": doc.metadata.get("is_table", False),
            "is_text": doc.metadata.get("is_text", False),
        }
        for doc in retrieved_docs
    ]
}

# 4. 평가 데이터 만들기
eval_rows = []

for item in eval_questions:
    result = answer_with_rag(item["question"])

    eval_rows.append({
        "question": item["question"],
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": item["ground_truth"],
        "page_info": result["page_info"],
    })

eval_df = pd.DataFrame(eval_rows)

display(eval_df[["question", "answer", "ground_truth", "page_info"]])

# 5. RAGAS 평가용 Dataset 변환
ragas_dataset = Dataset.from_pandas(
    eval_df[["question", "answer", "contexts", "ground_truth"]]
)

# 6. 성능평가 실행
eval_llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=0
)

score = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy],  # 2개만
    llm=eval_llm,
    embeddings=embeddings,
)

score_df = score.to_pandas()

display(score_df)

# 7. 평균 점수 확인
metric_cols = [
    col for col in [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
    if col in score_df.columns
]

mean_scores = score_df[metric_cols].mean()

print("=== 평균 성능 점수 ===")
print(mean_scores)

C:\Users\user\AppData\Local\Temp\ipykernel_18712\2598903737.py:13: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_18712\2598903737.py:13: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_18712\2598903737.py:13: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\user\AppData\Local\Temp\ipykernel_187

,question,answer,ground_truth,page_info
0,문서에서 소비자 문제 경험률이 높은 상위 지역은 어디인가요?,"- 소비자 문제 경험률 상위 지역: 경기, 서울권, 경기/인천권, 제주 \n- 경...",소비자 문제 경험률은 일부 지역에서 높게 나타난다,"[{'page': 56, 'has_visual_hint': True, 'is_tab..."
1,그래프나 표에서 확인되는 소비자 피해 증가 추세는 무엇인가요?,"- 소비자문제 경험률 전반적 하락 \n- 연령별 50대, 60대 이상, 20대에서...",소비자 문제 경험률은 시기별로 증가하거나 감소하는 변동 추세를 보인다,"[{'page': 55, 'has_visual_hint': True, 'is_tab..."
2,소비자가 피해를 예방하기 위해 주의해야 할 점은 무엇인가요?,- 신뢰할 수 있는 정보 확인 \n- 상품 및 서비스 품질 점검 \n- 배송 상...,"계약 조건 확인, 과장 광고 주의, 환불 및 교환 조건 확인이 중요하다","[{'page': 49, 'has_visual_hint': True, 'is_tab..."
3,소비자 상담이나 피해구제와 관련해 문서가 강조하는 내용은 무엇인가요?,- 소비자 문제 경험률 감소 추세 \n- 상품·서비스 품질불량 주요 문제 \n-...,소비자 상담과 피해구제는 소비자 문제 해결과 분쟁 조정에 중요한 역할을 한다,"[{'page': 57, 'has_visual_hint': True, 'is_tab..."
4,이 문서를 절약 코칭 서비스에 활용한다면 어떤 피드백에 쓸 수 있나요?,- 소비자 문제 경험률 기반 피드백 \n- 상품·서비스 품질불량 주의 \n- 가...,불필요한 소비를 줄이고 계약 전 확인을 강화하며 충동구매를 예방하는 데 활용할 수 있다,"[{'page': 57, 'has_visual_hint': True, 'is_tab..."


Evaluating: 100%|██████████| 10/10 [00:33<00:00,  3.30s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,문서에서 소비자 문제 경험률이 높은 상위 지역은 어디인가요?,[PART 1_제6장 2023 한국의 소비생활 변화와 전망 827 ○ (거래유형별)...,"- 소비자 문제 경험률 상위 지역: 경기, 서울권, 경기/인천권, 제주 \n- 경...",소비자 문제 경험률은 일부 지역에서 높게 나타난다,1.000000,0.544709
1,그래프나 표에서 확인되는 소비자 피해 증가 추세는 무엇인가요?,[| --- | --- | --- | --- | --- | --- | --- | -...,"- 소비자문제 경험률 전반적 하락 \n- 연령별 50대, 60대 이상, 20대에서...",소비자 문제 경험률은 시기별로 증가하거나 감소하는 변동 추세를 보인다,1.000000,0.271398
2,소비자가 피해를 예방하기 위해 주의해야 할 점은 무엇인가요?,[816 2023 한국의 소비생활지표 | 구분 | 구분 | 표본수 | 경험(%) |...,- 신뢰할 수 있는 정보 확인 \n- 상품 및 서비스 품질 점검 \n- 배송 상...,"계약 조건 확인, 과장 광고 주의, 환불 및 교환 조건 확인이 중요하다",0.571429,0.387187
3,소비자 상담이나 피해구제와 관련해 문서가 강조하는 내용은 무엇인가요?,[| 구분 | 표본수 | 소비자 문제 경험률 | 상품·서비스 품질불량 | 상품·서비...,- 소비자 문제 경험률 감소 추세 \n- 상품·서비스 품질불량 주요 문제 \n-...,소비자 상담과 피해구제는 소비자 문제 해결과 분쟁 조정에 중요한 역할을 한다,0.875000,0.304293
4,이 문서를 절약 코칭 서비스에 활용한다면 어떤 피드백에 쓸 수 있나요?,[| 구분 | 표본수 | 소비자 문제 경험률 | 상품·서비스 품질불량 | 상품·서비...,- 소비자 문제 경험률 기반 피드백 \n- 상품·서비스 품질불량 주의 \n- 가...,불필요한 소비를 줄이고 계약 전 확인을 강화하며 충동구매를 예방하는 데 활용할 수 있다,0.941176,0.153533


=== 평균 성능 점수 ===
faithfulness        0.877521
answer_relevancy    0.332224
dtype: float64
